# Notebook 4 — SPRINGLab F5-Hindi-24KHz

- Repo (SPRINGLab fork): https://github.com/rumourscape/F5-TTS
- Model: https://huggingface.co/SPRINGLab/F5-Hindi-24KHz
- 151 M params, F5 "small" config, **Hindi-only training**. Roman / English will be heavily OOD; record failures, don't paper over.

> The exact CLI flags and checkpoint filenames may have shifted since this notebook was written.
> **Open the model card and `/tmp/f5_hi/README.md` first** — copy whatever the canonical example currently says.


In [ ]:
# === Cell 1: clone repo + install ===
!rm -rf /tmp/f5_hi
!git clone --depth 1 https://github.com/rumourscape/F5-TTS /tmp/f5_hi
%cd /tmp/f5_hi
!pip install -q -e .
!pip install -q huggingface_hub soundfile


In [ ]:
# === Cell 2: download checkpoint ===
from huggingface_hub import snapshot_download
import os
model_dir = snapshot_download("SPRINGLab/F5-Hindi-24KHz", local_dir="/tmp/f5hi_ckpt")
# Inspect what was downloaded so we can wire up the right --ckpt_file path.
print(os.listdir(model_dir))


In [ ]:
# Mount Drive (or skip if running locally) and clone the audit folder.
# Adjust this cell to point AUDIT_DIR at wherever audit/ lives in your runtime.
import os
from pathlib import Path

# Two common patterns:
#   1. Colab + Drive: AUDIT_DIR = "/content/drive/MyDrive/hienglish/audit"
#   2. Colab + git clone:
#         !git clone https://github.com/<you>/hienglish.git /content/hienglish
#         AUDIT_DIR = "/content/hienglish/audit"
#   3. Local: AUDIT_DIR = str(Path.cwd().parent / "audit")  (if launched from notebooks/)

AUDIT_DIR = os.environ.get("AUDIT_DIR", "/content/audit")
assert Path(AUDIT_DIR).is_dir(), f"AUDIT_DIR={AUDIT_DIR} missing — set it before running."
print(f"AUDIT_DIR = {AUDIT_DIR}")


In [ ]:
import csv
from pathlib import Path

EVAL_TSV = Path(AUDIT_DIR) / "eval_sentences.tsv"
with open(EVAL_TSV, encoding="utf-8") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 30, f"expected 30 sentences, got {len(rows)}"
print(f"Loaded {len(rows)} sentences from {EVAL_TSV}")
print(rows[0])


In [ ]:
# === Cell 5: prepare paths + reference clip ===
from pathlib import Path
import time, json, subprocess

MODEL_NAME = "springlab_f5"
OUT = Path(AUDIT_DIR) / "results" / MODEL_NAME
OUT.mkdir(parents=True, exist_ok=True)

REF_AUDIO = str(Path(AUDIT_DIR) / "reference_audio" / "hindi_ref.wav")
with open(Path(AUDIT_DIR) / "reference_audio" / "hindi_ref.txt", encoding="utf-8") as f:
    REF_TEXT = f.read().strip()

# Pick the right safetensors / .pt file. The canonical name on the model card is
# `model_2500000.safetensors` but verify against the directory listing above.
import glob
ckpt_candidates = glob.glob(f"{model_dir}/model_*.safetensors") or glob.glob(f"{model_dir}/model_*.pt")
assert ckpt_candidates, "No SPRINGLab checkpoint file found"
CKPT = ckpt_candidates[0]
VOCAB = f"{model_dir}/vocab.txt"
print(f"CKPT={CKPT}\nVOCAB={VOCAB}")


In [ ]:
# === Cell 6: run inference via the F5-TTS CLI ===
# This is the documented invocation in the SPRINGLab fork. If the fork changes,
# fall back to the Python API:
#     from f5_tts.api import F5TTS
#     tts = F5TTS(model="F5TTS_Base", ckpt_file=CKPT, vocab_file=VOCAB, use_ema=True, device="cuda")
#     wav, sr, _ = tts.infer(ref_file=REF_AUDIO, ref_text=REF_TEXT, gen_text=text,
#                            nfe_step=32, cfg_strength=2.0, sway_sampling_coef=-1.0,
#                            speed=1.0, remove_silence=True, seed=42, file_wave=str(out_path))
log = []
for r in rows:
    rid, cat, text = r["id"], r["category"], r["text"]
    t0 = time.time()
    out_path = OUT / f"{rid}.wav"
    try:
        cmd = [
            "f5-tts_infer-cli",
            "--model", "F5TTS_Base",
            "--ckpt_file", CKPT,
            "--vocab_file", VOCAB,
            "--ref_audio", REF_AUDIO,
            "--ref_text", REF_TEXT,
            "--gen_text", text,
            "--output_dir", str(OUT),
            "--output_file", out_path.name,
            "--nfe_step", "32", "--cfg_strength", "2.0", "--speed", "1.0",
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
        if result.returncode != 0 or not out_path.exists():
            raise RuntimeError(result.stderr[-500:])
        log.append({"id": rid, "category": cat, "elapsed_s": time.time() - t0, "status": "ok"})
    except Exception as e:
        log.append({"id": rid, "category": cat, "status": "error", "error": str(e)})
        print(f"  [error] {rid}: {e}")


In [ ]:
import json
out_log = Path(OUT) / "log.json"
with open(out_log, "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)

n_ok = sum(1 for x in log if x["status"] == "ok")
print(f"{MODEL_NAME}: {n_ok}/30 succeeded — log at {out_log}")
